# 2장 — BPE 토크나이저 (실습)

교재 `docs/book/02-bpe-tokenizer.md` 와 함께 본다. 이 노트북에서 하는 것:

1. BPE 알고리즘을 장난감 문자열에서 손으로 따라가기
2. 바이트 단위 BPE 를 한글 코퍼스에 그대로 적용하면 무슨 일이 생기는지 (GPT 가 한글에 불리한 이유)
3. "글자 먼저 조립" 으로 고친 뒤, 어휘 크기에 따라 시퀀스 길이가 어떻게 줄어드는지
4. 프로젝트 토크나이저 `data/tokenizers/bpe-8192.json` 저장

> 전체 실행 약 3~4분. 학습 셀 두 개(§2, §3)가 각각 1분 남짓이다.

## 1. 알고리즘 — 가장 잦은 쌍을 합친다, 반복한다

In [ ]:
import time

from shllm.config import TOKENIZER_DIR, ensure_data_dirs
from shllm.data import load_corpus
from shllm.tokenizer import BPETokenizer, CharTokenizer

ensure_data_dirs()
toy = "aaabdaaabac"
tok = BPETokenizer.train(toy, vocab_size=256 + 3, char_first=False)
for i, (a, b) in enumerate(tok.merges):
    print(f"병합 {i}: ({tok.token_str(a)!r}, {tok.token_str(b)!r}) → 토큰 {256 + i} = {tok.token_str(256 + i)!r}")
print("인코딩:", [tok.token_str(i) for i in tok.encode(toy)])

가장 잦은 쌍 `aa` 가 토큰 256, 다음 `ab` 가 257, 그 다음은 두 토큰을 합친 `aaab` 가 258 이 된다. 인코딩은 **배운 순서대로** 병합을 적용한다.
`char_first=False` 는 순수 바이트 BPE (GPT-2 방식). 한글에서 이걸 그대로 쓰면 어떻게 되는지 본다.

## 2. 바이트 BPE 를 한글에 그대로 쓰면

한글 한 글자는 UTF-8 로 **3바이트**다. 바이트 256개에서 출발하면 "옛" 하나를 만드는 데도 병합 두 번이 필요하다.
문제는 병합 순서가 **빈도순**이라는 것 — "옛" 의 마지막 바이트와 "날" 의 첫 바이트가 자주 붙어 나오면 그 쌍이 먼저 합쳐진다.

In [ ]:
text = load_corpus("korean-classics")
print(f"{len(text):,} 자 = {len(text.encode('utf-8')):,} 바이트  (한글 1자 ≈ 3바이트)")

t0 = time.perf_counter()
byte_bpe = BPETokenizer.train(text, vocab_size=4096, char_first=False)
print(f"바이트 BPE 4096 학습: {time.perf_counter() - t0:.0f}초")

In [ ]:
sample = "옛날 옛적에 호랑이가 담배 피우던 시절에, 김첨지는 운수 좋은 날이라고 생각하였다."
print([byte_bpe.token_str(i) for i in byte_bpe.encode(sample)])

In [ ]:
def partial_tokens(tok: BPETokenizer) -> int:
    """온전한 글자로 디코딩되지 않는(바이트가 잘린) 토큰 수"""
    return sum(1 for i in range(256, tok.vocab_size) if "\ufffd" in tok.vocab[i].decode("utf-8", errors="replace"))


ids = byte_bpe.encode(text)
print(f"어휘 4096 중 잘린 바이트 토큰: {partial_tokens(byte_bpe)}개")
print(f"코퍼스 → {len(ids):,} 토큰, 토큰당 {len(text) / len(ids):.2f} 자")

`<0xEC><0x98>` + `<0x9B>` 처럼 "옛" 이 **두 조각**으로 남았고, 병합 3,840개 중 1/6 이 글자 경계에 걸린 바이트 조각이다.
모델 입장에서는 "옛" 이라는 개념이 두 토큰에 걸쳐 있고, 앞뒤 글자에 따라 조각이 달라진다.
실제 GPT-2·GPT-3 토크나이저로 한글을 넣으면 글자당 2~3 토큰이 나오는 것이 정확히 이 현상이다 — 같은 내용을 영어보다 2~3배 비싸게 처리한다.

## 3. 고치기 — 글자를 먼저 조립한다

빈도 병합에 앞서, 코퍼스에 2번 이상 나온 글자를 바이트에서 통째로 만드는 병합을 먼저 넣는다 (`char_first=True`, 기본값).
그 뒤 빈도 병합은 **온전한 글자끼리만** 합치게 된다. 드문 글자(한 번 나온 한자 등)는 여전히 바이트로 남아 — 어떤 입력이든 인코딩은 된다.

In [ ]:
t0 = time.perf_counter()
bpe = BPETokenizer.train(text, vocab_size=8192, verbose=True)
print(f"글자 우선 BPE 8192 학습: {time.perf_counter() - t0:.0f}초")
print([bpe.token_str(i) for i in bpe.encode(sample)])

In [ ]:
# 잘린 토큰 = 여러 글자가 공유하는 앞 2바이트 (예: <0xEC><0x98> 은 '옛','오','온'… 의 공통 접두). 글자 경계를 넘는 조각은 없다
print(f"어휘 8192 중 잘린 바이트 토큰: {partial_tokens(bpe)}개 (모두 글자 조립의 중간 단계)")
seq = bpe.encode(text)
print(f"코퍼스 → {len(seq):,} 토큰, 토큰당 {len(text) / len(seq):.2f} 자")

### 3.1 어휘 크기 vs 시퀀스 길이

병합은 순서대로 쌓이므로, 앞쪽 k 개만 남기면 어휘 크기 k+256 짜리 토크나이저가 된다 (`truncated`). 다시 학습할 필요 없이 어휘 크기 실험을 한다.

In [ ]:
import matplotlib
import matplotlib.pyplot as plt
from matplotlib import font_manager

cjk = [f.name for f in font_manager.fontManager.ttflist if "CJK" in f.name]
if cjk:
    matplotlib.rcParams["font.family"] = cjk[0]

char_tok = CharTokenizer.from_text(text)
rows = [("char", char_tok.vocab_size, len(text))]
for k in (1024, 2048, 4096, 8192):
    small = bpe.truncated(k)
    rows.append((f"bpe {k}", k, len(small.encode(text))))
for name, v, n in rows:
    print(f"{name:>9}: 어휘 {v:>5,}  토큰 {n:>10,}  토큰당 {len(text) / n:.2f} 자")

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot([r[1] for r in rows[1:]], [r[2] / 1000 for r in rows[1:]], "o-", label="BPE")
ax.axhline(len(text) / 1000, ls="--", c="gray", label=f"글자 단위 (V={char_tok.vocab_size:,})")
ax.set_xscale("log", base=2)
ax.set_xlabel("어휘 크기 V")
ax.set_ylabel("코퍼스 토큰 수 (천)")
ax.legend()
plt.show()

어휘 2,048 까지는 병합 예산이 글자 조립(약 2,500개)에 다 쓰여 사실상 글자 단위와 같다. 그 뒤로 어휘를 두 배 늘릴 때마다 시퀀스가 짧아지지만 체감은 줄어든다.

이것이 **토크나이저의 트레이드오프**다:

- 어휘 V 가 크면 → 시퀀스 T 가 짧다 (어텐션 비용은 T² — 5장) · 토큰 하나에 더 많은 뜻 · 그러나 임베딩 표 V×C 가 커지고 드문 토큰은 학습 기회가 적다
- 어휘 V 가 작으면 → 그 반대

GPT-2 는 50,257, 최근 모델은 10만~25만을 쓴다. 우리는 코퍼스가 114만 자라 8,192 로 시작하고 9장에서 비교한다.

## 4. 어떤 입력이든 — 처음 보는 글자·이모지

In [ ]:
weird = "龍이 🐯를 만났다. Hello, world!"
ids = bpe.encode(weird)
print([bpe.token_str(i) for i in ids])
print("복원:", bpe.decode(ids), "| 일치:", bpe.decode(ids) == weird)
try:
    char_tok.encode(weird)
except KeyError as e:
    print("CharTokenizer 는 처음 보는 글자에서 KeyError:", e)

코퍼스에 없던 `🐯` 는 4바이트로 쪼개져 들어가고 그대로 복원된다. **바이트가 바닥에 있으면 모르는 글자가 없다** — BPE 가 바이트에서 출발하는 이유다.

## 5. 저장 — 이후 장의 프로젝트 토크나이저

In [ ]:
path = TOKENIZER_DIR / "bpe-8192.json"
bpe.save(path)
loaded = BPETokenizer.load(path)
assert loaded.encode(sample) == bpe.encode(sample)
print(path, f"({path.stat().st_size / 1024:.0f} KB, 병합 {len(loaded.merges):,}개)")

## 정리

- BPE = "가장 잦은 인접 쌍을 새 토큰으로" 를 반복. 어휘 크기를 마음대로 정하고, 바이트에서 출발하므로 모르는 글자가 없다.
- 한글은 글자당 3바이트라 **빈도순 병합이 글자 경계를 무시**한다 → 글자를 먼저 조립하면 온전한 글자 단위의 어휘가 된다.
- 어휘 크기 ↔ 시퀀스 길이 트레이드오프. 이 프로젝트는 `bpe-8192.json` 으로 시작한다.
- 이제 텍스트는 `[2237, 1421, ...]` 같은 정수 열이다. 이 정수를 신경망이 다룰 **벡터** 로 바꾸는 것이  3장이다.

---
**다음 장**: 3장 — 임베딩. 토큰 id 를 학습 가능한 벡터로.